# Deep Learning Approach to fraud detection of job listings (BERT)

# Loading Libraries

In [ ]:
import re
import pandas as pd
import numpy as np
import evaluate
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## Data Preperations

In [41]:
df = pd.read_csv("data/DataSet.csv")
df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,f,f
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,f,f


##  Build a single “text” field + label‑encode

In [42]:
# removing HTML tags to clean dataset
def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text()

text_cols = [
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

for col in text_cols:
    df[col] = df[col].apply(clean_html)

/var/folders/q2/_kj_jvmd09d2vj26tyjy25nh0000gn/T/ipykernel_17161/96894343.py:5: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  return BeautifulSoup(text, "html.parser").get_text()


In [43]:
def clean_html(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return re.sub(r"<[^>]+>", " ", text).replace("\n", " ").strip()

def build_text(row):
    parts = [
        row.get("title", ""),
        row.get("company_profile", ""),
        row.get("description", ""),
        row.get("requirements", ""),
    ]
    text = " \n ".join([clean_html(p) for p in parts if isinstance(p, str) and p.strip()])
    return text

df["text"] = df.apply(build_text, axis=1)
df["label"] = df["fraudulent"].map({"t": 1, "f": 0})
df = df[df["text"].str.len() > 10]  # drop empties

In [44]:
df

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset,text,label
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",\nExperience with content management systems a...,,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f,"Marketing Intern \n We're Food52, and we've cr...",0
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:\nYour key responsibil...,What you will get from us\nThrough being part ...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f,Customer Service - Cloud Video Production \n 9...,0
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,\nValor Services provides Workforce Solutions ...,"Our client, located in Houston, is actively se...",\nImplement pre-commissioning and commissionin...,,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f,Commissioning Machinery Assistant (CMA) \n Val...,0
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"\n\nEDUCATION: Bachelor’s or Master’s in GIS, ...",Our culture is anything but corporate—we have ...,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,f,f,Account Executive - Washington DC \n Our passi...,0
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review Manager\nLOCATIO...,QUALIFICATIONS:\n\nRN license in the State of ...,Full Benefits Offered,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,f,f,Bill Review Manager \n SpotSource Solutions LL...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17875,Account Director - Distribution,"CA, ON, Toronto",Sales,NaN,Vend is looking for some awesome new talent to...,Just in case this is the first time you’ve vis...,To ace this role you:\n\nWill eat comprehensiv...,What can you expect from us?\nWe have an open ...,f,t,t,Full-time,Mid-Senior level,NaN,Computer Software,Sales,f,f,Account Director - Distribution \n Vend is loo...,0
17876,Payroll Accountant,"US, PA, Philadelphia",Accounting,NaN,WebLinc is the e-commerce platform and service...,\nThe Payroll Accountant will focus primarily ...,\n- B.A. or B.S. in Accounting\n- Desire to ha...,\nHealth & Wellness\n\nMedical plan\nPrescript...,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Internet,Accounting/Auditing,f,f,Payroll Accountant \n WebLinc is the e-commerc...,0
17877,Project Cost Control Staff Engineer - Cost Con...,"US, TX, Houston",NaN,NaN,We Provide Full Time Permanent Positions for m...,Experienced Project Cost Control Staff Enginee...,\nAt least 12 years professional experience.\n...,,f,f,f,Full-time,NaN,NaN,NaN,NaN,f,f,Project Cost Control Staff Engineer - Cost Con...,0
17878,Graphic Designer,"NG, LA, Lagos",NaN,NaN,,Nemsia Studios is looking for an experienced v...,1. Must be fluent in the latest versions of Co...,Competitive salary (compensation will be based...,f,f,t,Contract,Not Applicable,Professional,Graphic Design,Design,f,f,Graphic Designer \n Nemsia Studios is looking ...,0


## Train / validation / test split (stratified)

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=42
)

print("train", len(train_df), "val", len(val_df), "test", len(test_df))

train 12918 val 2280 test 2682


## Create Hugging Face datasets + tokenize with BERT

In [ ]:
model_name = "bert-base-uncased"   
tokenizer = AutoTokenizer.from_pretrained(model_name)

def prepare_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]])
    return ds

train_ds = prepare_dataset(train_df)
val_ds   = prepare_dataset(val_df)
test_ds  = prepare_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/12918 [00:00<?, ? examples/s]

Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/2682 [00:00<?, ? examples/s]

## Train BERT classifier (Trainer)

In [ ]:
pipeline_type = "bert"

# Local model path
model_path = "model/bert"

# Load tokenizer and model from local directory
# The model appears to be a base BERT checkpoint without a classification head, so
# fine-tuning will train the missing classifier weights.
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

# Metrics for Trainer

def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions

    preds = np.argmax(logits, axis=1)
    # In case logits is shape (batch, 1) (unexpected) guard for indexing
    probs = logits[:, 1] if logits.shape[-1] > 1 else logits[:, 0]

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0,
    }

training_args = TrainingArguments(
    output_dir="outputs/bert-fraud",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: model/bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your do

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.067003,0.080067,0.983772,0.962025,0.690909,0.804233,0.956900


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1615, training_loss=0.11967878493627435, metrics={'train_runtime': 6413.5923, 'train_samples_per_second': 2.014, 'train_steps_per_second': 0.252, 'total_flos': 1699434306570240.0, 'train_loss': 0.11967878493627435, 'epoch': 1.0})

## Evaluate on the test set

In [ ]:
predictions = trainer.predict(test_ds)
logits = predictions.predictions
pred_labels = np.argmax(logits, axis=1)
probs = logits[:, 1]  # probability of fraud

# Add predictions back to a DataFrame
test_df["predicted_label"] = pred_labels
test_df["fraud_prob"] = probs

# Save predictions
test_df.to_csv("data/bert_test_predictions.csv", index=False)

/opt/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
